# prompts

In [ ]:
"""
prompts class 계층도

BasePromptTemplate --> PipelinePromptTemplate
                       StringPromptTemplate --> PromptTemplate
                                                FewShotPromptTemplate
                                                FewShotPromptWithTemplates
                       BaseChatPromptTemplate --> AutoGPTPrompt
                                                  ChatPromptTemplate --> AgentScratchPadChatPromptTemplate



BaseMessagePromptTemplate --> MessagesPlaceholder
                              BaseStringMessagePromptTemplate --> ChatMessagePromptTemplate
                                                                  HumanMessagePromptTemplate
                                                                  AIMessagePromptTemplate
                                                                  SystemMessagePromptTemplate
"""
None

# API Key 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
print(f'{os.environ['OPENAI_API_KEY'][:20]}...')

sk-proj-iKU13YeoxNgF...


# 기본 import

In [3]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai.llms.base import OpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

In [4]:
chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# PromptTemplate

In [6]:
# 방식1
t = PromptTemplate.from_template("What is the capital of {country}")
t.format(country='France')

'What is the capital of France'

In [7]:
# 방식2
t2 = PromptTemplate(
    template="What is the capital of {country}",
    input_variables=['country'],   # 입력변수들을 PromptTemplate 에 알려주어야 한다.
)

t2.format(country='France')

'What is the capital of France'

# FewShotPromptTemplate
모델에 예제(example) 주기

In [ ]:
# 모델에게 '어떻게 대답해야 하는 지에 대한 예제(example)'를 AI 모델에게 주는 것이
# prompt 를 사용해서 '어떻게 대답해야 하는지 알려주는 것'보다 훨씬 좋다

# FewShotPromptTemplate 이 하는 일이 바로 그거다!
# - 이를 통해 예제(샘플)를 형식화(포맷) 할수 있다.
# - 이런 예제들을 데이터베이스등에 저장시켜놓고 활용할수도 있다

# 예시들을 활용하는 예
# 고객지원 봇.
#   다른 고객들ㅇ과의 대화 기록, 많은 기록들...
#   LLM 에게 '어떻게 고객에게 대응' 하는지 알려줄때.

In [8]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate

In [9]:
#예제들 (examples) 준비
# 모델더러 나에게 '이런 식ㅇ로 답벼해 줬으면 좋겟다' 라고 제시.

examples = [
    {
       "question": "What do you know about France?",

        # 원하는 형식의 답변..
        "answer": """
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro        
        """,
    },
  {
    "question": "What do you know about Italy?",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "question": "What do you know about Greece?",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },
    
]


In [10]:
# 일단 예제 없이 호출하면?
chat.invoke("What do you know about France?")

France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral.

France is the largest country in the European Union by land area and the third-largest in Europe overall. It has a population of around 67 million people. The official language is French.

France is a popular tourist destination, attracting millions of visitors each year to its beautiful cities, countryside, and coastline. The country is also known for its wine and cheese production, fashion industry, and art and literature.

France has a long history of political and cultural influence, including the French Revolution, which had a significant impact on the development of modern democracy. It is a founding member of the United Nations, NATO, and the European Union.

Overall, France is a diverse and vibrant country with a rich cultural heritage and a strong influ

AIMessage(content='France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral.\n\nFrance is the largest country in the European Union by land area and the third-largest in Europe overall. It has a population of around 67 million people. The official language is French.\n\nFrance is a popular tourist destination, attracting millions of visitors each year to its beautiful cities, countryside, and coastline. The country is also known for its wine and cheese production, fashion industry, and art and literature.\n\nFrance has a long history of political and cultural influence, including the French Revolution, which had a significant impact on the development of modern democracy. It is a founding member of the United Nations, NATO, and the European Union.\n\nOverall, France is a diverse and vibrant country with a rich cultural 

In [11]:
# FewShotPromptTemplate 사용

# 포맷 준비 <- exmples 로 포맷팅할거다
example_template = """
    Human: {question}
    AI: {answer}
"""
# ↑ {question} 과 {answer} 는 위 examples 와 동일한 key 를 사용하여 작성해두어야 한다.


In [12]:
example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n')

In [13]:
print(example_prompt.format(**examples[2]))
# print(example_prompt.format(
#     answer = examples[2]['answer'],
#     question = examples[2]['question']
# ))


    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      



In [14]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,  # 사용할 prompt
    examples=examples,   # 준비된 예제들

    # 유저의 질문을 넣어주자
    suffix="Human: What do you know abount {country}?",   # 질문은 example 과 동일한 형식
    input_variables=['country']
)

prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n          Here is what I know:\n          Capital: Paris\n          Language: French\n          Food: Wine and Cheese\n          Currency: Euro        \n        '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know abount {country}?')

In [15]:
print(prompt.format(country='Germany'))


    Human: What do you know about France?
    AI: 
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro        
        



    Human: What do you know about Italy?
    AI: 
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      



    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      


Human: What do you know abount Germany?


In [ ]:
# step1  example 리스트를 만들고   examples
# step2  FewShotPromptTemplate 에 전달했고 examples=
# step3  어떻게 전달한 예제들을 형식화 할지 알려주었고
# step4  마지막에 질문을 포함시켰다.  suffix, input_variables

# AI 는 우리의 예제들과 똑같은 구조, 형태로 답변하게 될겁니다


In [16]:
chain = prompt | chat

chain

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n          Here is what I know:\n          Capital: Paris\n          Language: French\n          Food: Wine and Cheese\n          Currency: Euro        \n        '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know abount {country}?')
| ChatOpenAI(callbacks=[<langchain_core.callbacks.streaming_stdout.Streaming

In [17]:
chain.invoke({
    "country": "Thailand",
})

AI: 
      I know this:
      Capital: Bangkok
      Language: Thai
      Food: Pad Thai and Tom Yum Goong
      Currency: Thai Baht

AIMessage(content='AI: \n      I know this:\n      Capital: Bangkok\n      Language: Thai\n      Food: Pad Thai and Tom Yum Goong\n      Currency: Thai Baht', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019dba55-a4ab-7651-a665-d2dd3337dd19', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 36, 'total_tokens': 189, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [18]:
chain.invoke({
    "country": "Turkey",
})

AI: 
      I know this:
      Capital: Ankara
      Language: Turkish
      Food: Kebab and Baklava
      Currency: Turkish Lira

AIMessage(content='AI: \n      I know this:\n      Capital: Ankara\n      Language: Turkish\n      Food: Kebab and Baklava\n      Currency: Turkish Lira', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019dba56-5e2e-7340-8149-84eb1aedb61e', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 35, 'total_tokens': 188, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# FewShotChatMessagePromptTemplate

In [19]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

In [20]:
examples = [
    {
       "country": "France",
        "answer": """
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro        
        """,
    },
  {
    "country": "Italy",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "country": "Greece",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },
    
]

In [21]:
example_prompt = ChatPromptTemplate.from_messages([
    ('human', 'What do you know abount {country}?'),  # 반드시 example 과 동일한 key 로!
    ('ai', '{answer}')
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_prompt,
    examples=examples,
    # suffix= 는 필요없다.
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert, you give short answers."),
    example_prompt,  # !!
    ('human', 'What do you know about {country}?'),  # example_prompt 의 human 메세지와 동일한 포맷
            # 이래서 suffix= 는 필요 없던 것이다.
])


In [22]:
final_prompt.format_messages(country='Germany')

[SystemMessage(content='You are a geography expert, you give short answers.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What do you know abount France?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n          Here is what I know:\n          Capital: Paris\n          Language: French\n          Food: Wine and Cheese\n          Currency: Euro        \n        ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know abount Italy?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know abount Greece?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      I know this:\n      Capital: Athens\n      L

In [23]:
chain = final_prompt | chat

chain.invoke({"country": "Germany"})


      I know this:
      Capital: Berlin
      Language: German
      Food: Bratwurst and Sauerkraut
      Currency: Euro

AIMessage(content='\n      I know this:\n      Capital: Berlin\n      Language: German\n      Food: Bratwurst and Sauerkraut\n      Currency: Euro', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019dba62-5ab3-7303-af29-9c62c5e88815', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 33, 'total_tokens': 203, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [24]:
chain.invoke({"country": "South Korea"})


      I know this:
      Capital: Seoul
      Language: Korean
      Food: Kimchi and Bibimbap
      Currency: South Korean Won

AIMessage(content='\n      I know this:\n      Capital: Seoul\n      Language: Korean\n      Food: Kimchi and Bibimbap\n      Currency: South Korean Won', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019dba62-c9f6-7280-9cc8-fe5b998f815d', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 171, 'output_tokens': 32, 'total_tokens': 203, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})